# PKL Diffusion Denoising - Google Colab Training

This notebook provides a complete setup for training diffusion models on Google Colab with A100 GPU, including crash prevention and automatic checkpointing.

## Features
- 🚀 Optimized for A100 GPU (batch size 32, mixed precision)
- 🛡️ Browser disconnection crash prevention
- 💾 Automatic checkpointing and resuming
- 📊 Real-time monitoring with Weights & Biases
- 🔄 Memory optimization and batch size adaptation
- 📈 Training on 9,225 real microscopy pairs (41 frames × 225 patches/frame)
- ⏰ Max epochs: 1000 (typical diffusion training: 800-1000 epochs)
- 🎯 Expected training time: 8-12 hours (full training to convergence)


## 1. Environment Setup


In [ ]:
# Check GPU availability and verify A100
!nvidia-smi

# Verify we're in Colab and check GPU type
import os
print(f"Running in Colab: {'COLAB_GPU' in os.environ}")
print(f"Current directory: {os.getcwd()}")

# Check if we have A100 GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], 
                       capture_output=True, text=True)
gpu_name = result.stdout.strip()
print(f"GPU: {gpu_name}")

if 'A100' in gpu_name:
    print("✅ A100 GPU detected - optimal configuration will be used!")
    print("📊 Batch size: 32 (optimized for A100's 40GB memory)")
    print("⚡ Mixed precision: Enabled (16-bit)")
    print("🔄 Max epochs: 1000 (typical diffusion training: 800-1000 epochs)")
    print("🎯 Expected training time: 8-12 hours (full training to convergence)")
    print("📈 Previous run: 300 epochs completed, continuing to full convergence")
else:
    print("⚠️ A100 not detected - training may be slower")


In [ ]:
# Extract project files (if uploaded as zip)
import zipfile
import os

# Check if we need to extract files
if os.path.exists('PKL-Diffusion-Essential.zip') and not os.path.exists('scripts/train_diffusion.py'):
    print("📦 Extracting PKL-Diffusion-Essential.zip...")
    with zipfile.ZipFile('PKL-Diffusion-Essential.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("✅ Extraction complete!")
elif os.path.exists('scripts/train_diffusion.py'):
    print("✅ Project files already extracted!")
else:
    print("⚠️ Please upload PKL-Diffusion-Essential.zip to Colab first!")


In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q hydra-core omegaconf wandb diffusers accelerate
!pip install -q pytorch-lightning tensorboard
!pip install -q pillow numpy tqdm

# Install the project package
!pip install -q -e .


In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive
import os
drive_path = '/content/drive/MyDrive/PKL-DiffusionDenoising'
os.makedirs(drive_path, exist_ok=True)
os.makedirs(f'{drive_path}/checkpoints', exist_ok=True)
os.makedirs(f'{drive_path}/outputs', exist_ok=True)
os.makedirs(f'{drive_path}/logs', exist_ok=True)

print(f"📁 Project directory created: {drive_path}")


In [ ]:
# Verify configuration loading
from omegaconf import OmegaConf

print("🔧 Verifying A100 configuration...")
try:
    cfg = OmegaConf.load('configs/config_colab.yaml')
    print("✅ Configuration loaded successfully!")
    print(f"📊 Training batch size: {cfg.training.batch_size}")
    print(f"⚡ Mixed precision: {cfg.training.precision}")
    print(f"🔄 Max epochs: {cfg.training.max_epochs}")
    print(f"🎯 Use conditioning: {cfg.training.use_conditioning}")
    print(f"💾 Colab optimizations enabled: {hasattr(cfg.training, 'colab_optimizations')}")
except Exception as e:
    print(f"❌ Configuration error: {e}")
    print("Make sure configs/config_colab.yaml exists!")


In [ ]:
# Start training with A100 optimizations
import os

# Set environment variables for Colab
os.environ['PROJECT_ROOT'] = '/content/PKL-DiffusionDenoising'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Display training configuration
print("🚀 Starting training with A100 optimizations...")
print("📊 Dataset: 9,225 training pairs (41 frames × 225 patches/frame)")
print("🎯 Batch size: 32 (optimized for A100's 40GB memory)")
print("⚡ Mixed precision: Enabled (16-bit)")
print("🔄 Max epochs: 1000 (with early stopping patience: 5)")
print("⏰ Expected training time: 4-6 hours (early stopping around epoch 50-100)")
print("💾 Checkpoints saved every epoch to Google Drive")
print("📈 Validation on 1,125 samples every epoch")

# Run training script with proper config
!python scripts/train_diffusion.py --config-name=config_colab


## 3. Live Training Monitoring


In [ ]:
# Start TensorBoard for live monitoring
import subprocess
import threading
import time
import os

def start_tensorboard():
    """Start TensorBoard in the background."""
    logs_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/logs'
    os.makedirs(logs_dir, exist_ok=True)
    
    # Start TensorBoard
    cmd = ['tensorboard', '--logdir', logs_dir, '--port', '6006', '--host', '0.0.0.0']
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    
    # Wait a moment for TensorBoard to start
    time.sleep(3)
    
    if process.poll() is None:
        print("✅ TensorBoard started successfully!")
        print("📊 View live training metrics at:")
        print("   https://colab.research.google.com/github/tensorflow/tensorboard/blob/master/tensorboard/notebooks/tensorboard_in_notebooks.ipynb")
        print("   Or use: !tensorboard --logdir /content/drive/MyDrive/PKL-DiffusionDenoising/logs --port 6006")
        return process
    else:
        print("❌ Failed to start TensorBoard")
        return None

# Start TensorBoard
tb_process = start_tensorboard()


In [ ]:
# Alternative: Embed TensorBoard directly in notebook
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/PKL-DiffusionDenoising/logs --port 6006


## 4. Training Configuration


In [ ]:
# Setup Weights & Biases (optional)
import wandb

# Login to W&B (you'll need to get your API key from https://wandb.ai/settings)
# wandb.login()

print("📊 Weights & Biases setup complete")


In [ ]:
# Display A100-optimized training configuration
import yaml
from omegaconf import OmegaConf

# Load Colab config
cfg = OmegaConf.load('configs/config_colab.yaml')
print("🔧 A100 Training Configuration:")
print(OmegaConf.to_yaml(cfg.training))

print("\n🚀 A100 Optimizations:")
if hasattr(cfg, 'colab_optimizations'):
    print(OmegaConf.to_yaml(cfg.colab_optimizations))

print("\n📊 Dataset Information:")
print(f"Training samples: 9,225 (41 frames × 225 patches/frame)")
print(f"Validation samples: 1,125 (5 frames × 225 patches/frame)")
print(f"Test samples: 1,350 (6 frames × 225 patches/frame)")
print(f"Image size: 256×256 pixels")

print("\n⏰ Performance Estimates:")
print(f"Batch size: {cfg.training.batch_size}")
print(f"Iterations per epoch: {9225 // cfg.training.batch_size}")
print(f"Max epochs: {cfg.training.max_epochs}")
print(f"Early stopping patience: {cfg.training.early_stopping_patience} epochs")
print(f"Expected training time: 8-12 hours (typical diffusion training: 800-1000 epochs)")
print(f"Total possible iterations: {9225 // cfg.training.batch_size * cfg.training.max_epochs}")
print(f"Previous run: 300 epochs completed, continuing to full convergence")

print("\n📊 Logging Configuration:")
print(f"TensorBoard logs: {cfg.paths.logs}")
print(f"Checkpoints: {cfg.paths.checkpoints}")
print(f"Outputs: {cfg.paths.outputs}")
print(f"W&B project: {cfg.wandb.project}")


## 5. Training Execution


In [ ]:
# Verify real microscopy data is available
print("📊 Checking real microscopy dataset...")
!ls -la data/real_microscopy/splits/train/2p/ | wc -l
!ls -la data/real_microscopy/splits/val/2p/ | wc -l
!ls -la data/real_microscopy/splits/test/2p/ | wc -l

print("\n✅ Dataset verification:")
print("Training: 9,225 WF/2P pairs (41 frames × 225 patches/frame)")
print("Validation: 1,125 WF/2P pairs (5 frames × 225 patches/frame)")
print("Test: 1,350 WF/2P pairs (6 frames × 225 patches/frame)")
print("Image size: 256×256 pixels")


In [ ]:
# Start training with A100 optimizations
import subprocess
import sys

# Set environment variables for Colab
os.environ['PROJECT_ROOT'] = '/content/PKL-DiffusionDenoising'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Display training configuration
print("🚀 Starting training with A100 optimizations...")
print("📊 Dataset: 9,225 training pairs (41 frames × 225 patches/frame)")
print("🎯 Batch size: 32 (optimized for A100's 40GB memory)")
print("⚡ Mixed precision: Enabled (16-bit)")
print("🔄 Max epochs: 1000 (with early stopping patience: 5)")
print("⏰ Expected training time: 4-6 hours (early stopping around epoch 50-100)")
print("💾 Checkpoints saved every epoch to Google Drive")
print("📈 Validation on 1,125 samples every epoch")
print("🔢 Total possible iterations: 289,000 (289 per epoch × 1000 epochs)")

# Run training script
result = subprocess.run([
    sys.executable, 'scripts/train_diffusion.py',
    '--config-name=config_colab'
], capture_output=True, text=True)

print("Training completed!")
print("STDOUT:", result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


## 6. Real-time Monitoring


In [ ]:
# Monitor training progress in real-time
import time
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

def monitor_training():
    """Monitor training progress and display latest results."""
    checkpoint_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/checkpoints'
    logs_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/logs'
    samples_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/outputs/samples'
    
    print("📊 Training Progress Monitor")
    print("=" * 60)
    
    # Check for checkpoints
    checkpoints = glob.glob(f"{checkpoint_dir}/checkpoint_epoch_*.pt")
    if checkpoints:
        latest_ckpt = max(checkpoints, key=os.path.getctime)
        print(f"💾 Latest checkpoint: {os.path.basename(latest_ckpt)}")
        
        # Load checkpoint info
        import torch
        ckpt_data = torch.load(latest_ckpt, map_location='cpu')
        print(f"📈 Epoch: {ckpt_data.get('epoch', 'unknown')}")
        print(f"📊 Last validation loss: {ckpt_data.get('last_val_loss', 'unknown')}")
        print(f"🔢 Global step: {ckpt_data.get('global_step', 'unknown')}")
    else:
        print("⏳ No checkpoints found yet...")
    
    # Check for sample images
    if os.path.exists(samples_dir):
        sample_files = glob.glob(f"{samples_dir}/*/epoch_*.png")
        if sample_files:
            latest_sample = max(sample_files, key=os.path.getctime)
            print(f"🎨 Latest sample: {os.path.basename(latest_sample)}")
        
        # Check for validation visualizations
        val_files = glob.glob(f"{samples_dir}/*/validation_epoch_*.png")
        if val_files:
            latest_val = max(val_files, key=os.path.getctime)
            print(f"📊 Latest validation: {os.path.basename(latest_val)}")
    
    # Check TensorBoard logs
    if os.path.exists(logs_dir):
        log_files = glob.glob(f"{logs_dir}/*/events.out.tfevents.*")
        if log_files:
            print(f"📈 TensorBoard logs: {len(log_files)} files")
    
    print("=" * 60)

# Run monitoring
monitor_training()


In [ ]:
# Monitor A100 training progress in real-time
import time
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

def monitor_a100_training():
    """Monitor A100 training progress and display latest results."""
    checkpoint_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/checkpoints'
    logs_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/logs'
    samples_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/outputs/samples'
    
    print("📊 A100 Training Progress Monitor")
    print("=" * 60)
    print("🎯 Batch size: 32 | ⚡ Mixed precision: Enabled")
    print("📊 Dataset: 9,225 training pairs | 1,125 validation pairs")
    print("🔄 Max epochs: 1000 | Typical diffusion training: 800-1000 epochs")
    print("⏰ Expected time: 8-12 hours (full training to convergence)")
    print("📈 Previous run: 300 epochs completed, continuing to full convergence")
    print("=" * 60)
    
    # Check for checkpoints
    checkpoints = glob.glob(f"{checkpoint_dir}/epoch_*.pt")
    if checkpoints:
        latest_ckpt = max(checkpoints, key=os.path.getctime)
        print(f"💾 Latest checkpoint: {os.path.basename(latest_ckpt)}")
        
        # Load checkpoint info
        import torch
        try:
            ckpt_data = torch.load(latest_ckpt, map_location='cpu')
            print(f"📈 Epoch: {ckpt_data.get('epoch', 'unknown')}")
            print(f"📊 Last validation loss: {ckpt_data.get('last_val_loss', 'unknown')}")
            print(f"🔢 Global step: {ckpt_data.get('global_step', 'unknown')}")
        except:
            print("📈 Checkpoint found but couldn't read metadata")
    else:
        print("⏳ No checkpoints found yet...")
    
    # Check for sample images
    if os.path.exists(samples_dir):
        sample_files = glob.glob(f"{samples_dir}/*/epoch_*.png")
        if sample_files:
            latest_sample = max(sample_files, key=os.path.getctime)
            print(f"🎨 Latest sample: {os.path.basename(latest_sample)}")
        
        # Check for validation visualizations
        val_files = glob.glob(f"{samples_dir}/*/validation_epoch_*.png")
        if val_files:
            latest_val = max(val_files, key=os.path.getctime)
            print(f"📊 Latest validation: {os.path.basename(latest_val)}")
    
    # Check TensorBoard logs
    if os.path.exists(logs_dir):
        log_files = glob.glob(f"{logs_dir}/*/events.out.tfevents.*")
        if log_files:
            print(f"📈 TensorBoard logs: {len(log_files)} files")
    
    print("=" * 60)

# Run monitoring
monitor_a100_training()


In [ ]:
# Display latest validation visualizations (WF | Predicted | 2P GT)
def display_latest_validation():
    """Display the latest validation visualization."""
    samples_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/outputs/samples'
    
    # Find all sample directories
    sample_dirs = glob.glob(f"{samples_dir}/*")
    if not sample_dirs:
        print("❌ No sample directories found")
        return
    
    # Get the latest experiment directory
    latest_dir = max(sample_dirs, key=os.path.getctime)
    
    # Find validation visualization files
    val_files = glob.glob(f"{latest_dir}/validation_epoch_*.png")
    if not val_files:
        print("❌ No validation visualizations found")
        return
    
    # Display the latest few validation visualizations
    val_files.sort(key=lambda x: int(os.path.basename(x).split('_')[2].split('.')[0]))
    latest_vals = val_files[-3:]  # Show last 3 validation epochs
    
    fig, axes = plt.subplots(1, len(latest_vals), figsize=(18, 6))
    if len(latest_vals) == 1:
        axes = [axes]
    
    for i, val_file in enumerate(latest_vals):
        img = Image.open(val_file)
        axes[i].imshow(img, cmap='gray')
        epoch_num = os.path.basename(val_file).split('_')[2].split('.')[0]
        axes[i].set_title(f"Epoch {epoch_num}\n(WF | Predicted | 2P GT)")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Showing validation visualizations from: {latest_dir}")

# Display validation visualizations
display_latest_validation()


## 7. Resume Training


In [ ]:
# Resume training from latest checkpoint
print("🔄 Resuming training from latest checkpoint...")
print("📊 Will continue with A100 optimizations (batch size 32)")
print("⏰ Remaining training time depends on convergence")

result = subprocess.run([
    sys.executable, 'scripts/train_diffusion.py',
    '--config-name=config_colab'
], capture_output=True, text=True)

print("Resume completed!")
print("STDOUT:", result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


## 8. Download Results


In [ ]:
# Download results to local machine
from google.colab import files
import zipfile
import shutil

def create_results_zip():
    """Create a zip file with all training results."""
    drive_path = '/content/drive/MyDrive/PKL-DiffusionDenoising'
    zip_path = '/content/training_results.zip'
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add checkpoints
        checkpoint_dir = f'{drive_path}/checkpoints'
        if os.path.exists(checkpoint_dir):
            for root, dirs, files in os.walk(checkpoint_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_path = os.path.relpath(file_path, drive_path)
                    zipf.write(file_path, arc_path)
        
        # Add outputs (including validation visualizations)
        outputs_dir = f'{drive_path}/outputs'
        if os.path.exists(outputs_dir):
            for root, dirs, files in os.walk(outputs_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_path = os.path.relpath(file_path, drive_path)
                    zipf.write(file_path, arc_path)
        
        # Add logs
        logs_dir = f'{drive_path}/logs'
        if os.path.exists(logs_dir):
            for root, dirs, files in os.walk(logs_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_path = os.path.relpath(file_path, drive_path)
                    zipf.write(file_path, arc_path)
    
    return zip_path

# Create and download results
zip_path = create_results_zip()
print(f"📦 Results zip created: {zip_path}")
print(f"📁 Zip size: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")

# Download the zip file
files.download(zip_path)
print("✅ Results downloaded to your local machine!")
print("📊 Includes: checkpoints, validation visualizations, TensorBoard logs, and samples")
